<a href="https://colab.research.google.com/github/noor-omar/DECI-colabProject/blob/main/EYOUTH_31005041700343_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Name: Noor Omar

ID: EYOUTH_31005041700343

Now, let's load data from `library.db`. We need the `checkouts` table and the `members` table from this database.

In [1]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect('/content/library.db')

# Load the 'checkouts' table into a DataFrame
df_checkouts_db = pd.read_sql_query("SELECT * FROM checkouts;", conn)
print("Checkouts from library.db (df_checkouts_db):")
display(df_checkouts_db.head())

# Load the 'members' table into a DataFrame
df_members_db = pd.read_sql_query("SELECT * FROM members;", conn)
print("\nMembers from library.db (df_members_db):")
display(df_members_db.head())

# Close the database connection
conn.close()

Checkouts from library.db (df_checkouts_db):


,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03



Members from library.db (df_members_db):


,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05


Next, let's load the book details from `books.json`.

In [2]:
# Load book details from books.json
df_books_json = pd.read_json('/content/books.json')
print("\nBooks from books.json (df_books_json):")
display(df_books_json.head())


Books from books.json (df_books_json):


,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books


Finally, let's load the additional checkouts data from `summer_checkouts.html`. We'll need to inspect the HTML structure to correctly extract the table.

In [3]:
# Load additional checkouts from summer_checkouts.html
# pandas can read HTML tables directly
df_checkouts_html = pd.read_html('/content/summer_checkouts.html')[0] # Assuming the first table is the one we need
print("\nCheckouts from summer_checkouts.html (df_checkouts_html):")
display(df_checkouts_html.head())


Checkouts from summer_checkouts.html (df_checkouts_html):


,Member ID,Book ID,Checkout Date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


Now, let's combine all the data sources. First, I'll standardize the column names in `df_checkouts_html` and then concatenate it with `df_checkouts_db` to get a unified checkouts list. After that, I'll merge with book details and member details.

In [4]:
# Standardize column names in df_checkouts_html to match df_checkouts_db
df_checkouts_html_renamed = df_checkouts_html.rename(columns={
    'Member ID': 'member_id',
    'Book ID': 'book_id',
    'Checkout Date': 'checkout_date'
})

# Select relevant columns from df_checkouts_db for concatenation
# Note: df_checkouts_db has 'checkout_id' and 'return_date' which df_checkouts_html_renamed doesn't
# We will align on common columns or create NaNs where data is missing
common_checkout_cols = ['member_id', 'book_id', 'checkout_date']

# Combine the two checkouts dataframes
df_all_checkouts = pd.concat([
    df_checkouts_db[common_checkout_cols],
    df_checkouts_html_renamed[common_checkout_cols]
], ignore_index=True)

print("Combined Checkouts Data (df_all_checkouts):")
display(df_all_checkouts.head())
print(f"Total checkouts after combining: {len(df_all_checkouts)}")


Combined Checkouts Data (df_all_checkouts):


,member_id,book_id,checkout_date
0,1047,517,2024-10-21
1,1072,513,2025-08-24
2,1053,523,2024-02-04
3,1032,513,2025-06-21
4,1079,511,2025-11-11


Total checkouts after combining: 417


Next, let's merge the combined checkouts with the `df_books_json` to add book details.

In [5]:
# Merge with book details (df_books_json)
df_combined = pd.merge(df_all_checkouts, df_books_json, on='book_id', how='left')

print("Combined Data with Book Details (df_combined):")
display(df_combined.head())


Combined Data with Book Details (df_combined):


,member_id,book_id,checkout_date,genre,pages,publication_year,publisher
0,1047,517,2024-10-21,Mystery,338,2015.0,Delta House
1,1072,513,2025-08-24,Science,294,2021.0,Oasis Books
2,1053,523,2024-02-04,Historical,276,2018.0,Oasis Books
3,1032,513,2025-06-21,Science,294,2021.0,Oasis Books
4,1079,511,2025-11-11,Historical,117,2016.0,Nile Press


Finally, I'll merge the dataset with `df_members_db` to include member details.

In [6]:
# Merge with member details (df_members_db)
df_final_checkouts = pd.merge(df_combined, df_members_db, on='member_id', how='left')

print("Final Combined Checkouts Dataset (df_final_checkouts):")
display(df_final_checkouts.head())
print(f"Total records in final dataset: {len(df_final_checkouts)}")

Final Combined Checkouts Dataset (df_final_checkouts):


,member_id,book_id,checkout_date,genre,pages,publication_year,publisher,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1047,517,2024-10-21,Mystery,338,2015.0,Delta House,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25
1,1072,513,2025-08-24,Science,294,2021.0,Oasis Books,Seif,Zaki,9.0,Zamalek,Active,2025-10-21
2,1053,523,2024-02-04,Historical,276,2018.0,Oasis Books,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03
3,1032,513,2025-06-21,Science,294,2021.0,Oasis Books,Nada,Zaki,7.0,Nasr City,Active,2025-10-19
4,1079,511,2025-11-11,Historical,117,2016.0,Nile Press,Rana,Osman,8.0,Shubra,Active,2024-10-27


Total records in final dataset: 417


Now, let's calculate how many books each member has borrowed in total from the combined dataset.

In [7]:
# Calculate total books borrowed per member
df_member_borrow_counts = df_final_checkouts.groupby('member_id').size().reset_index(name='total_books_borrowed')

print("Total books borrowed per member:")
display(df_member_borrow_counts.head())


Total books borrowed per member:


,member_id,total_books_borrowed
0,1001,1
1,1002,3
2,1003,10
3,1005,4
4,1006,1


Finally, I'll save the complete `df_final_checkouts` DataFrame to a CSV file named `task1_combined_data.csv`.

In [8]:
# Save the combined DataFrame to a CSV file
df_final_checkouts.to_csv('task1_combined_data.csv', index=False)

print("DataFrame saved to 'task1_combined_data.csv'")

DataFrame saved to 'task1_combined_data.csv'


To incorporate the `total_books_borrowed` count into the main dataset, I will merge `df_member_borrow_counts` with `df_final_checkouts`.

In [9]:
# Merge total_books_borrowed into the df_final_checkouts
df_final_checkouts_with_borrow_counts = pd.merge(
    df_final_checkouts,
    df_member_borrow_counts,
    on='member_id',
    how='left'
)

print("Combined DataFrame with Total Books Borrowed:")
display(df_final_checkouts_with_borrow_counts.head())

Combined DataFrame with Total Books Borrowed:


,member_id,book_id,checkout_date,genre,pages,publication_year,publisher,first_name,last_name,grade,neighborhood,membership_status,join_date,total_books_borrowed
0,1047,517,2024-10-21,Mystery,338,2015.0,Delta House,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16
1,1072,513,2025-08-24,Science,294,2021.0,Oasis Books,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14
2,1053,523,2024-02-04,Historical,276,2018.0,Oasis Books,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5
3,1032,513,2025-06-21,Science,294,2021.0,Oasis Books,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6
4,1079,511,2025-11-11,Historical,117,2016.0,Nile Press,Rana,Osman,8.0,Shubra,Active,2024-10-27,10


Now, I will save this updated DataFrame to `task1_combined_data.csv`.

In [10]:
# Save the updated combined DataFrame to a CSV file
df_final_checkouts_with_borrow_counts.to_csv('task1_combined_data.csv', index=False)

print("Updated DataFrame saved to 'task1_combined_data.csv'")

Updated DataFrame saved to 'task1_combined_data.csv'


Let's identify the columns with missing values in our `df_final_checkouts_with_borrow_counts` DataFrame.

In [11]:
# Calculate missing values
missing_values = df_final_checkouts_with_borrow_counts.isnull().sum()
missing_percentage = (df_final_checkouts_with_borrow_counts.isnull().sum() / len(df_final_checkouts_with_borrow_counts)) * 100

# Create a DataFrame to display missing values
missing_info = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})

# Filter to show only columns with missing values and sort by percentage
missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("Columns with Missing Values:")
display(missing_info)

Columns with Missing Values:


,Missing Count,Missing Percentage
grade,41,9.832134
publication_year,35,8.393285
join_date,11,2.637890
last_name,5,1.199041
first_name,5,1.199041
neighborhood,5,1.199041
membership_status,5,1.199041


Based on your request, I will now fill the missing values for `neighborhood`, `first_name`, and `last_name` columns with the string 'Unknown'.

In [12]:
# Fill missing values for specified columns with 'Unknown'
df_final_checkouts_with_borrow_counts['neighborhood'] = df_final_checkouts_with_borrow_counts['neighborhood'].fillna('Unknown')
df_final_checkouts_with_borrow_counts['first_name'] = df_final_checkouts_with_borrow_counts['first_name'].fillna('Unknown')
df_final_checkouts_with_borrow_counts['last_name'] = df_final_checkouts_with_borrow_counts['last_name'].fillna('Unknown')

print("Missing values filled for 'neighborhood', 'first_name', and 'last_name' with 'Unknown'.")

# Verify the changes by re-checking missing values for these columns
print("\nVerifying missing values after filling:")
missing_info_after_fill = df_final_checkouts_with_borrow_counts[['neighborhood', 'first_name', 'last_name']].isnull().sum()
display(missing_info_after_fill)

Missing values filled for 'neighborhood', 'first_name', and 'last_name' with 'Unknown'.

Verifying missing values after filling:


,0
neighborhood,0
first_name,0
last_name,0


Now, let's address the remaining missing values as per your instructions.

In [13]:
# 1. Fill missing 'grade' values with the median
median_grade = df_final_checkouts_with_borrow_counts['grade'].median()
df_final_checkouts_with_borrow_counts['grade'] = df_final_checkouts_with_borrow_counts['grade'].fillna(median_grade)
print(f"Missing 'grade' values filled with median: {median_grade}")

# 2. Fill missing 'publication_year' values with 1900
df_final_checkouts_with_borrow_counts['publication_year'] = df_final_checkouts_with_borrow_counts['publication_year'].fillna(1900)
print("Missing 'publication_year' values filled with 1900.")

# 3. Fill missing 'join_date' values with '1900-01-01'
# First, ensure 'join_date' is in datetime format to handle NaT if any
df_final_checkouts_with_borrow_counts['join_date'] = pd.to_datetime(df_final_checkouts_with_borrow_counts['join_date'], errors='coerce')
df_final_checkouts_with_borrow_counts['join_date'] = df_final_checkouts_with_borrow_counts['join_date'].fillna(pd.to_datetime('1900-01-01'))
print("Missing 'join_date' values filled with '1900-01-01'.")

# 4. Fill missing 'membership_status' values with the mode
mode_membership_status = df_final_checkouts_with_borrow_counts['membership_status'].mode()[0]
df_final_checkouts_with_borrow_counts['membership_status'] = df_final_checkouts_with_borrow_counts['membership_status'].fillna(mode_membership_status)
print(f"Missing 'membership_status' values filled with mode: {mode_membership_status}")

# Verify all missing values again
print("\nVerifying all missing values after filling:")
missing_info_final = df_final_checkouts_with_borrow_counts.isnull().sum()
missing_info_final = missing_info_final[missing_info_final > 0]

if missing_info_final.empty:
    print("All missing values have been handled.")
else:
    display(missing_info_final)


Missing 'grade' values filled with median: 8.0
Missing 'publication_year' values filled with 1900.
Missing 'join_date' values filled with '1900-01-01'.
Missing 'membership_status' values filled with mode: Active

Verifying all missing values after filling:
All missing values have been handled.


Now that missing values have been handled, let's check for and remove any duplicate records in the `df_final_checkouts_with_borrow_counts` DataFrame.

In [14]:
# Check for duplicate records
duplicate_rows = df_final_checkouts_with_borrow_counts.duplicated().sum()
print(f"Number of duplicate records found: {duplicate_rows}")

# Remove duplicate records
df_final_checkouts_cleaned = df_final_checkouts_with_borrow_counts.drop_duplicates()

print(f"Number of records after removing duplicates: {len(df_final_checkouts_cleaned)}")


Number of duplicate records found: 8
Number of records after removing duplicates: 409


### Problem 3: The Same Value Written Different Ways

This problem often arises in categorical columns where the same conceptual value might be represented in slightly different ways (e.g., 'Science Fiction' vs. 'science fiction', 'NY' vs. 'New York'). Let's identify such issues in our dataset, particularly in categorical columns like `genre`, `publisher`, `neighborhood`, and `membership_status`.

To fix values that might be written in different ways, I will perform the following steps:
1.  **Standardize text columns**: For all columns with `object` dtype (which typically holds strings), I will remove leading/trailing whitespace and convert the text to title case.
2.  **Consolidate 'genre' values**: I will map 'Science Fiction' to 'Science' as it appears to be a related category.

In [15]:
# To avoid SettingWithCopyWarning, explicitly work on a copy
df_final_checkouts_cleaned = df_final_checkouts_cleaned.copy()

# Identify all object (string) columns for general cleaning
text_columns = df_final_checkouts_cleaned.select_dtypes(include='object').columns

print(f"Applying general text cleaning (strip and title case) to columns: {list(text_columns)}\n")

for col in text_columns:
    df_final_checkouts_cleaned[col] = df_final_checkouts_cleaned[col].str.strip().str.title()

# Verify the changes in genre
print("\nUnique values in 'genre' column after cleaning:")
display(df_final_checkouts_cleaned['genre'].value_counts())

print("\nUnique values in 'publisher' column after cleaning:")
display(df_final_checkouts_cleaned['publisher'].value_counts())

print("\nUnique values in 'membership_status' column after cleaning:")
display(df_final_checkouts_cleaned['membership_status'].value_counts())


Applying general text cleaning (strip and title case) to columns: ['checkout_date', 'genre', 'publisher', 'first_name', 'last_name', 'neighborhood', 'membership_status']


Unique values in 'genre' column after cleaning:


,count
genre,
Science,112
Adventure,110
Friendship,61
Mystery,44
Historical,32
Nature,23
Science Fiction,21
Poetry,6



Unique values in 'publisher' column after cleaning:


,count
publisher,
Nile Press,203
Oasis Books,112
Cairo Young Readers,53
Delta House,41



Unique values in 'membership_status' column after cleaning:


,count
membership_status,
Active,325
Inactive,84


### Summary of Fixes for 'The Same Value Written Different Ways'

*   **General Text Cleaning**: For all string-based columns (`genre`, `publisher`, `first_name`, `last_name`, `neighborhood`, `membership_status`, `checkout_date`), leading/trailing whitespace has been removed, and values have been converted to title case to ensure consistency.
*   **Genre Handling**: The 'Science Fiction' and 'Science' genres have been kept as distinct categories, as per your instruction.

This process helps in standardizing categorical data, making it more uniform and reliable for analysis.

In [16]:
# Save the cleaned DataFrame to a new CSV file
df_final_checkouts_cleaned.to_csv('task2_cleaned_data.csv', index=False)

print("Cleaned DataFrame saved to 'task2_cleaned_data.csv'")

Cleaned DataFrame saved to 'task2_cleaned_data.csv'
